# 01_data_sharing_agreement_<agreement> — governance control plane

Use this governance-owned notebook to define and approve agreement metadata once, then reuse the approved metadata across Sandbox, Dev-Test, and Prod notebooks.

- Canonical notebook pattern: `01_data_sharing_agreement_<agreement>`.
- This template file can stay named `01_data_agreement_template`, but implementations should follow the canonical naming pattern.
- Governance truth is cross-environment and steward-owned, not environment-local.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit import (
    build_governance_context,
    review_governance,
    write_governance,
    load_governance,
    prepare_governance_input,
    suggest_pii_labels,
    extract_pii_suggestions,
    write_metadata_rows,
    get_path,
)


In [ ]:
# Runtime placeholders (lowercase style)
env_name = "dev"
dataset_name = "<agreement_name>"
table_name = "<table_name>"

# Agreement metadata placeholders
agreement_id = "<agreement_id>"
agreement_name = "<agreement_name>"
agreement_context = "<agreement_context>"
approved_usage = "<approved_usage>"
business_context = "<business_context>"
ownership = "<ownership>"
permissions = "<permissions>"
restrictions = "<restrictions>"
classification = "<classification_posture>"
sensitivity = "<sensitivity_posture>"
pii = "<pii_posture>"
related_notebook_links = ["<related_notebook_link>"]

# Optional extras
data_owner = "<data_owner>"
governance_steward = "<governance_steward>"
approved_dq_metadata_notes_or_links = "<approved_dq_metadata_notes_or_links>"
related_02_ex_notebooks = ["<02_ex_agreement_topic_notebook>"]
related_03_pc_notebooks = ["<03_pc_agreement_pipeline_notebook>"]

governance_prompt_context = build_governance_context(
    business_context=business_context,
    approved_usage=approved_usage,
    dataset_context=agreement_context,
    steward_notes=(
        f"Agreement: {agreement_name}; Owner: {data_owner}; Steward: {governance_steward}; "
        f"classification: {classification}; sensitivity: {sensitivity}; pii: {pii}; "
        f"permissions: {permissions}; restrictions: {restrictions}; DQ refs: {approved_dq_metadata_notes_or_links}; "
        f"related notebooks: {related_notebook_links}"
    ),
)

agreement_context_metadata = {
    "agreement_id": agreement_id,
    "agreement_context": agreement_context,
    "approved_usage": approved_usage,
    "business_context": business_context,
    "ownership": ownership,
    "permissions": permissions,
    "restrictions": restrictions,
    "classification": classification,
    "sensitivity": sensitivity,
    "pii": pii,
    "related_notebook_links": ", ".join(related_notebook_links),
    "agreement_name": agreement_name,
    "governance_prompt_context": governance_prompt_context,
    "data_owner": data_owner,
    "governance_steward": governance_steward,
    "approved_dq_metadata_notes_or_links": approved_dq_metadata_notes_or_links,
    "related_02_ex_notebooks": ", ".join(related_02_ex_notebooks),
    "related_03_pc_notebooks": ", ".join(related_03_pc_notebooks),
}
save_to_metadata = False


## Optional AI-assisted governance suggestions (advisory only)

Use AI suggestions only when helpful. Suggestions are advisory and do not approve controls.

Human approval is required before approved governance metadata is written.


In [ ]:
# Optional: use profile + approved business context when available.
# Example source table names can be replaced by your metadata table names.
use_ai_profile_input = False
if use_ai_profile_input:
    profile_rows = [r.asDict(recursive=True) for r in spark.table("METADATA_PROFILE_ROWS").filter(f"table_name = '{table_name}'").collect()]
    approved_business_context_rows = [r.asDict(recursive=True) for r in spark.table("METADATA_COLUMN_CONTEXT").filter("status = 'approved'").collect()]
    prepared_rows = prepare_governance_input(profile_rows, table_name=table_name, column_contexts=approved_business_context_rows)
    prepared_df = spark.createDataFrame(prepared_rows)
    governance_response_df = suggest_pii_labels(
        prepared_profile_df=prepared_df,
        prompt=CONFIG.ai_prompt_config.governance_personal_identifier_template,
    )
    governance_suggestions = extract_pii_suggestions(governance_response_df)
else:
    governance_suggestions = [
        {"column_name": "customer_id", "approved_business_context": "Business identifier for customer records", "ai_suggested_personal_identifier_classification": "indirect_identifier", "confidentiality_label": "confidential"},
        {"column_name": "email", "approved_business_context": "Customer communication contact", "ai_suggested_personal_identifier_classification": "direct_identifier", "confidentiality_label": "restricted"},
    ]

governance_suggestions


In [ ]:
metadata_path = get_path(env_name, "metadata", config=FABRIC_CONFIG)

# Human-in-the-loop review is required before write-back.
review_governance(
    suggestions=governance_suggestions,
    environment_name=env_name,
    dataset_name=dataset_name,
    table_name=table_name,
)

print("Review complete. Only human-approved rows should be persisted.")


In [ ]:
if save_to_metadata:
    saved_rows = write_governance(
        spark,
        metadata_path=metadata_path,
        agreement_context=agreement_context_metadata,
        table_name="METADATA_COLUMN_GOVERNANCE",
        mode="append",
    )
    print(f"Saved governance rows: {saved_rows}")
else:
    print("Dry run: skipped write_governance.")


In [ ]:
# Optional agreement-level snapshot row for METADATA_DATA_AGREEMENT.
agreement_record = dict(agreement_context_metadata)

if save_to_metadata:
    write_metadata_rows(
        spark,
        rows=[agreement_record],
        metadata_path=metadata_path,
        table_name="METADATA_DATA_AGREEMENT",
        mode="append",
    )
    print("Saved agreement snapshot row.")


In [ ]:
# Downstream read-only context load (for 02_ex and 03_pc notebooks).
approved_governance_df = spark.table("METADATA_COLUMN_GOVERNANCE")
agreement_df = spark.table("METADATA_DATA_AGREEMENT")
read_only_context = load_governance(
    governance_rows=approved_governance_df,
    agreement_rows=agreement_df,
    agreement_id=agreement_id,
)
print("Loaded read-only approved governance context for downstream reuse.")


## Re-run guidance

- Keep `agreement_id` stable for the same agreement.
- `%run 00_env_config` is runtime bootstrap only; governance truth remains cross-environment and steward-owned.
- Sandbox, Dev-Test, and Prod notebooks reuse approved metadata from this agreement control plane.
- Keep DQ enforcement in downstream notebooks (`03_pc`), not in this notebook.
